In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/project_kar'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install numpy pandas scipy matplotlib netCDF4 xarray pykrige scikit-learn requests geopandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
import xarray as xr
import netCDF4 as nc
from pykrige.ok import OrdinaryKriging
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np


metadata = pd.read_excel('All_CONUS_Station_Information_V2.xlsx')

print(metadata.columns.tolist())
print(metadata.head())



df_temp   = pd.read_excel('USCRN_AirTemperature_2006_2021.xlsx')
df_precip = pd.read_excel('USCRN_Precipitation_2006_2021.xlsx')
df_rh     = pd.read_excel('USCRN_RelativeHumidity_2006_2021.xlsx')
df_soilm  = pd.read_excel('USCRN_SoilMoisture10cm_2006_2021.xlsx')
df_soilt  = pd.read_excel('USCRN_SoilTemperature_2006_2021.xlsx')


print(df_temp['date'].iloc[0])


print(df_temp['date'].iloc[0])


station_cols = [c for c in df_temp.columns if c != 'date']

df_temp[station_cols]  = df_temp[station_cols]  - 273.15
df_soilt[station_cols] = df_soilt[station_cols] - 273.15



def melt_to_long(df, var_name):
    station_cols = [c for c in df.columns if c != 'date']
    melted = df.melt(id_vars='date', value_vars=station_cols,
                     var_name='StationName', value_name=var_name)
    return melted

long_temp   = melt_to_long(df_temp,   'air_temperature')
long_precip = melt_to_long(df_precip, 'precipitation')
long_rh     = melt_to_long(df_rh,     'relative_humidity')
long_soilm  = melt_to_long(df_soilm,  'soil_moisture')
long_soilt  = melt_to_long(df_soilt,  'soil_temperature')


station_data = long_temp.copy()
station_data = station_data.merge(long_precip, on=['date', 'StationName'], how='left')
station_data = station_data.merge(long_rh,     on=['date', 'StationName'], how='left')
station_data = station_data.merge(long_soilm,  on=['date', 'StationName'], how='left')
station_data = station_data.merge(long_soilt,  on=['date', 'StationName'], how='left')


station_data = station_data.merge(
    metadata[['StationName', 'Lon', 'Lat']],
    on='StationName', how='left'
)


station_data = station_data.rename(columns={'Lon': 'lon', 'Lat': 'lat'})


station_data.loc[station_data['relative_humidity'] > 100, 'relative_humidity'] = np.nan
station_data.loc[station_data['precipitation'] < 0, 'precipitation'] = np.nan

print(f"\nDate range: {station_data['date'].min()} to {station_data['date'].max()}")
print(f"Total rows: {len(station_data)}")
print(f"Stations: {station_data['StationName'].nunique()}")
print(station_data.head())
print(station_data[['air_temperature','precipitation',
                     'relative_humidity','soil_moisture',
                     'soil_temperature']].describe())

['StationName', 'StationID', 'Lon', 'Lat']
                          StationName  StationID     Lon    Lat
0                  AK_Fairbanks_11_NE      26494 -147.51  64.97
1                       AK_Sitka_1_NE      25379 -135.33  57.06
2                    AK_St._Paul_4_NE      25711 -170.21  57.16
3  AK_Utqiaġvik_formerly_Barrow_4_ENE      27516 -156.61  71.32
4                     AL_Gadsden_19_N      63857  -85.96  34.29
2006-01-01 00:00:00
2006-01-01 00:00:00

Date range: 2006-01-01 00:00:00 to 2021-12-31 00:00:00
Total rows: 1367496
Stations: 234
        date         StationName  air_temperature  precipitation  \
0 2006-01-01  AK_Fairbanks_11_NE            -10.1            0.0   
1 2006-01-02  AK_Fairbanks_11_NE            -13.5            0.0   
2 2006-01-03  AK_Fairbanks_11_NE            -13.6            0.0   
3 2006-01-04  AK_Fairbanks_11_NE            -14.9            0.0   
4 2006-01-05  AK_Fairbanks_11_NE            -16.4            0.0   

   relative_humidity  soil_moistur

In [ ]:
# SAVE
station_data.to_parquet(f'{folder_path}/station_data.parquet')
print("✓ station_data saved!")

✓ station_data saved!


In [ ]:


lon_min, lon_max = -125, -67
lat_min, lat_max = 24, 50
resolution = 2.0  # degrees

grid_lons = np.arange(lon_min + resolution/2, lon_max, resolution)
grid_lats = np.arange(lat_min + resolution/2, lat_max, resolution)

grid_lon_2d, grid_lat_2d = np.meshgrid(grid_lons, grid_lats)

print(f"Grid size: {len(grid_lats)} rows × {len(grid_lons)} cols")
print(f"Total grid cells: {len(grid_lats) * len(grid_lons)}")

Grid size: 13 rows × 29 cols
Total grid cells: 377


In [ ]:
import numpy as np

# SAVE
np.save(f'{folder_path}/grid_lons.npy', grid_lons)
np.save(f'{folder_path}/grid_lats.npy', grid_lats)
np.save(f'{folder_path}/grid_lon_2d.npy', grid_lon_2d)
np.save(f'{folder_path}/grid_lat_2d.npy', grid_lat_2d)
print("✓ Grid arrays saved!")

✓ Grid arrays saved!


In [ ]:
from scipy.spatial import cKDTree

def idw_interpolation(lon_obs, lat_obs, values_obs, lon_grid, lat_grid, power=2):
    mask = ~np.isnan(values_obs)
    lon_obs = lon_obs[mask]
    lat_obs = lat_obs[mask]
    values_obs = values_obs[mask]

    if len(values_obs) < 3:
        return np.full(lon_grid.shape, np.nan)

    obs_coords = np.column_stack([lon_obs, lat_obs])
    grid_shape = lon_grid.shape
    grid_coords = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])

    tree = cKDTree(obs_coords)
    k = min(len(values_obs), 12)
    distances, indices = tree.query(grid_coords, k=k)
    distances = np.where(distances == 0, 1e-10, distances)

    weights = 1.0 / distances**power
    weights_sum = weights.sum(axis=1)
    values_at_neighbors = values_obs[indices]
    grid_values_flat = (weights * values_at_neighbors).sum(axis=1) / weights_sum

    return grid_values_flat.reshape(grid_shape)


# ── Test for one day ──────────────────────────────────────────
target_date = pd.Timestamp('2010-07-15')   # must be within 2006–2021
variable = 'air_temperature'

day_data = station_data[station_data['date'] == target_date].copy()

print(f"Stations available on {target_date.date()}: {len(day_data)}")

lon_obs = day_data['lon'].values
lat_obs = day_data['lat'].values
val_obs = day_data[variable].values

grid_idw = idw_interpolation(lon_obs, lat_obs, val_obs, grid_lon_2d, grid_lat_2d, power=2)

print(f"IDW result for {target_date.date()}, {variable}")
print(f"  Min: {np.nanmin(grid_idw):.2f} °C")
print(f"  Max: {np.nanmax(grid_idw):.2f} °C")
print(f"  Mean: {np.nanmean(grid_idw):.2f} °C")

Stations available on 2010-07-15: 234
IDW result for 2010-07-15, air_temperature
  Min: 12.16 °C
  Max: 36.48 °C
  Mean: 24.91 °C


In [ ]:
from pykrige.ok import OrdinaryKriging
from scipy.spatial import cKDTree
from tqdm import tqdm
import pickle
import os
import numpy as np
import pandas as pd


date_range = pd.date_range(start='2006-01-01', end='2021-12-31', freq='D')
print(f"✓ date_range: {date_range[0].date()} to {date_range[-1].date()} "
      f"({len(date_range)} days)")


VARIOGRAM_PARAMS = {
    'spherical':   None,
    'exponential': None,
    'gaussian':    {'nugget': 0.5, 'sill': 1.0, 'range': 5.0},
}


def kriging_interpolation(lon_obs, lat_obs, values_obs, lon_grid, lat_grid,
                           variogram_model='spherical'):
    mask = ~np.isnan(values_obs)
    lon_obs    = lon_obs[mask]
    lat_obs    = lat_obs[mask]
    values_obs = values_obs[mask]

    if len(values_obs) < 5:
        return np.full(lon_grid.shape, np.nan)

    v_mean = np.mean(values_obs)
    v_std  = np.std(values_obs) if np.std(values_obs) > 0 else 1.0
    values_norm = (values_obs - v_mean) / v_std

    try:
        params = VARIOGRAM_PARAMS.get(variogram_model, None)
        OK = OrdinaryKriging(
            lon_obs, lat_obs, values_norm,
            variogram_model=variogram_model,
            variogram_parameters=params,
            verbose=False,
            enable_plotting=False,
            nlags=6,
            weight=True,
        )
        z, _ = OK.execute('grid', lon_grid[0, :], lat_grid[:, 0])
        return np.array(z) * v_std + v_mean

    except Exception as e:
        print(f"  Kriging [{variogram_model}] failed: {e}")
        return np.full(lon_grid.shape, np.nan)



variogram_models = ['spherical', 'exponential', 'gaussian']
variables = ['air_temperature', 'precipitation',
             'relative_humidity', 'soil_temperature', 'soil_moisture']

n_lats  = len(grid_lats)
n_lons  = len(grid_lons)
n_dates = len(date_range)

CHECKPOINT_FILE = f'{folder_path}/kriging_checkpoint.pkl'
SAVE_EVERY      = 100


if os.path.exists(CHECKPOINT_FILE):
    print("🔁 Checkpoint found — resuming from where it crashed...")
    with open(CHECKPOINT_FILE, 'rb') as f:
        checkpoint = pickle.load(f)
    results_kriging_all = checkpoint['results']
    start_idx           = checkpoint['last_saved_idx'] + 1
    print(f"   Resuming from day index {start_idx} / {n_dates}")
    print(f"   ({date_range[start_idx].date()} onwards)")
else:
    print("🆕 No checkpoint found — starting fresh...")
    results_kriging_all = {
        model: {var: np.full((n_dates, n_lats, n_lons), np.nan) for var in variables}
        for model in variogram_models
    }
    start_idx = 0


print(f"\nTotal days to process: {n_dates - start_idx} remaining\n")

for t_idx in tqdm(range(start_idx, n_dates), desc='Kriging all models'):

    date   = date_range[t_idx]
    day_df = station_data[station_data['date'] == date]

    if len(day_df) == 0:
        continue

    for var in variables:
        lon_obs = day_df['lon'].values
        lat_obs = day_df['lat'].values
        val_obs = day_df[var].values

        for model in variogram_models:
            results_kriging_all[model][var][t_idx] = kriging_interpolation(
                lon_obs, lat_obs, val_obs,
                grid_lon_2d, grid_lat_2d,
                variogram_model=model
            )


    if (t_idx + 1) % SAVE_EVERY == 0:
        checkpoint = {
            'results':        results_kriging_all,
            'last_saved_idx': t_idx,
        }
        with open(CHECKPOINT_FILE, 'wb') as f:
            pickle.dump(checkpoint, f)
        print(f"\n💾 Checkpoint saved at day {t_idx+1}/{n_dates} "
              f"({date_range[t_idx].date()})")


with open(f'{folder_path}/results_kriging_all.pkl', 'wb') as f:
    pickle.dump(results_kriging_all, f)

if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

print("\n✅ All Kriging models saved!")
print(f"   Models    : {variogram_models}")
print(f"   Variables : {variables}")
print(f"   Shape     : ({n_dates}, {n_lats}, {n_lons}) per variable")

✓ date_range: 2006-01-01 to 2021-12-31 (5844 days)
🆕 No checkpoint found — starting fresh...

Total days to process: 5844 remaining



Kriging all models:   2%|▏         | 101/5844 [00:34<56:31,  1.69it/s]  


💾 Checkpoint saved at day 100/5844 (2006-04-10)


Kriging all models:   3%|▎         | 200/5844 [01:14<1:20:14,  1.17it/s]


💾 Checkpoint saved at day 200/5844 (2006-07-19)


...


💾 Checkpoint saved at day 5700/5844 (2021-08-09)


Kriging all models:  99%|█████████▉| 5800/5844 [1:03:51<00:51,  1.16s/it]


💾 Checkpoint saved at day 5800/5844 (2021-11-17)


Kriging all models: 100%|██████████| 5844/5844 [1:04:29<00:00,  1.51it/s]



✅ All Kriging models saved!
   Models    : ['spherical', 'exponential', 'gaussian']
   Variables : ['air_temperature', 'precipitation', 'relative_humidity', 'soil_temperature', 'soil_moisture']
   Shape     : (5844, 13, 29) per variable


In [ ]:
from scipy.spatial import cKDTree
from tqdm import tqdm
import numpy as np
import pandas as pd
import pickle


date_range = pd.date_range(start='2006-01-01', end='2021-12-31', freq='D')
print(f"✓ date_range: {date_range[0].date()} to {date_range[-1].date()} ({len(date_range)} days)")


def idw_interpolation(lon_obs, lat_obs, values_obs, lon_grid, lat_grid, power=2):
    mask = ~np.isnan(values_obs)
    lon_obs    = lon_obs[mask]
    lat_obs    = lat_obs[mask]
    values_obs = values_obs[mask]

    if len(values_obs) < 3:
        return np.full(lon_grid.shape, np.nan)

    obs_coords  = np.column_stack([lon_obs, lat_obs])
    grid_coords = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])

    tree = cKDTree(obs_coords)
    k = min(len(values_obs), 12)
    distances, indices = tree.query(grid_coords, k=k)
    distances = np.where(distances == 0, 1e-10, distances)

    weights             = 1.0 / distances**power
    weights_sum         = weights.sum(axis=1)
    values_at_neighbors = values_obs[indices]
    grid_values_flat    = (weights * values_at_neighbors).sum(axis=1) / weights_sum

    return grid_values_flat.reshape(lon_grid.shape)



variables = ['air_temperature', 'precipitation',
             'relative_humidity', 'soil_temperature', 'soil_moisture']

n_lats  = len(grid_lats)
n_lons  = len(grid_lons)
n_dates = len(date_range)

results_idw = {
    var: np.full((n_dates, n_lats, n_lons), np.nan)
    for var in variables
}


for t_idx, date in enumerate(tqdm(date_range, desc='IDW interpolation')):

    day_df = station_data[station_data['date'] == date]

    if len(day_df) == 0:
        continue

    for var in variables:
        lon_obs = day_df['lon'].values
        lat_obs = day_df['lat'].values
        val_obs = day_df[var].values

        results_idw[var][t_idx] = idw_interpolation(
            lon_obs, lat_obs, val_obs,
            grid_lon_2d, grid_lat_2d,
            power=2
        )

print("\n✅ IDW interpolation done!")
print(f"   Variables : {variables}")
print(f"   Shape     : ({n_dates}, {n_lats}, {n_lons}) per variable")

✓ date_range: 2006-01-01 to 2021-12-31 (5844 days)


IDW interpolation: 100%|██████████| 5844/5844 [01:22<00:00, 71.24it/s]


✅ IDW interpolation done!
   Variables : ['air_temperature', 'precipitation', 'relative_humidity', 'soil_temperature', 'soil_moisture']
   Shape     : (5844, 13, 29) per variable


In [ ]:
import pickle

# Save IDW
with open(f'{folder_path}/results_idw.pkl', 'wb') as f:
    pickle.dump(results_idw, f)
print("✓ IDW saved!")

# Save all 3 Kriging models
with open(f'{folder_path}/results_kriging_all.pkl', 'wb') as f:
    pickle.dump(results_kriging_all, f)
print("✓ All Kriging models saved!")

✓ IDW saved!
✓ All Kriging models saved!


In [ ]:
import xarray as xr

def save_to_netcdf(results_dict, grid_lons, grid_lats, date_range,
                   method_name, output_path):
    data_vars = {}
    for var, arr in results_dict.items():
        data_vars[var] = xr.DataArray(
            data=arr,
            dims=['time', 'lat', 'lon'],
            attrs={
                'long_name': var.replace('_', ' ').title(),
                'interpolation_method': method_name,
                'source': 'USCRN station observations',
            }
        )
    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            'time': ('time', date_range),
            'lat':  ('lat', grid_lats),
            'lon':  ('lon', grid_lons),
        },
        attrs={
            'title': f'Gridded Hydroclimatic Variables — {method_name}',
            'spatial_resolution': '2 degree',
            'temporal_resolution': 'daily',
            'domain': 'Conterminous USA',
            'created_by': 'GNR 640 Mini Project',
        }
    )
    ds.to_netcdf(output_path)
    print(f"✓ Saved: {output_path}")
    return ds


# Save IDW
ds_idw = save_to_netcdf(
    results_idw, grid_lons, grid_lats, date_range,
    method_name='IDW_power2',
    output_path=f'{folder_path}/gridded_USA_IDW.nc'
)

# Save all 3 Kriging models
for model in ['spherical', 'exponential', 'gaussian']:
    save_to_netcdf(
        results_kriging_all[model], grid_lons, grid_lats, date_range,
        method_name=f'Kriging_{model}',
        output_path=f'{folder_path}/gridded_USA_Kriging_{model}.nc'
    )

print("\n✓ All files saved to Drive!")

✓ Saved: /content/drive/MyDrive/project_kar/gridded_USA_IDW.nc
✓ Saved: /content/drive/MyDrive/project_kar/gridded_USA_Kriging_spherical.nc
✓ Saved: /content/drive/MyDrive/project_kar/gridded_USA_Kriging_exponential.nc
✓ Saved: /content/drive/MyDrive/project_kar/gridded_USA_Kriging_gaussian.nc

✓ All files saved to Drive!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install cdsapi

In [ ]:

import os

api_key = "d191a4d0-15cf-469d-81b9-afbb16bb086d"   # ← paste your API key here

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(f"url: https://cds.climate.copernicus.eu/api\n")   # new URL
    f.write(f"key: {api_key}\n")                                # no UID prefix

print("Done!")

folder_path = '/content/drive/MyDrive/project_kar'
os.makedirs(folder_path, exist_ok=True)

print(f"Folder created: {folder_path}")

Done!
Folder created: /content/drive/MyDrive/project_kar


In [ ]:
import cdsapi
c = cdsapi.Client()
print("Connected!")

Connected!


In [ ]:
import cdsapi
import os

c = cdsapi.Client()

folder_path = '/content/drive/MyDrive/project_kar'

for year in range(2011, 2013):
    for month in range(1, 13):

        output_file = f'{folder_path}/era5_mean_{year}_{month:02d}.nc'

        if os.path.exists(output_file):
            print(f"Mean {year}-{month:02d} already exists, skipping...")
            continue

        print(f"Downloading mean variables — {year}-{month:02d}...")

        c.retrieve(
            'reanalysis-era5-land',
            {
                'variable': [
                    '2m_temperature',
                    '2m_dewpoint_temperature',
                    'soil_temperature_level_1',
                    'volumetric_soil_water_layer_1',
                ],
                'year':  [str(year)],
                'month': [f'{month:02d}'],
                'day':   [f'{d:02d}' for d in range(1, 32)],
                'time':  ['00:00', '06:00', '12:00', '18:00'],  # 4 per day
                'area':  [50, -125, 24, -67],
                'format': 'netcdf',
            },
            output_file
        )
        print(f"✓ Saved: era5_mean_{year}_{month:02d}.nc")

print("\n✅ All MEAN downloads done!\n")


for year in range(2011, 2013):
    for month in range(1, 13):

        output_file = f'{folder_path}/era5_precip_{year}_{month:02d}.nc'

        if os.path.exists(output_file):
            print(f"Precip {year}-{month:02d} already exists, skipping...")
            continue

        print(f"Downloading precipitation — {year}-{month:02d}...")

        c.retrieve(
            'reanalysis-era5-land',
            {
                'variable': ['total_precipitation'],
                'year':  [str(year)],
                'month': [f'{month:02d}'],
                'day':   [f'{d:02d}' for d in range(1, 32)],
                'time':  ['23:00'],   # accumulation since midnight = daily total
                'area':  [50, -125, 24, -67],
                'format': 'netcdf',
            },
            output_file
        )
        print(f"✓ Saved: era5_precip_{year}_{month:02d}.nc")

print("\n✅ All PRECIP downloads done!")
print("\nSummary:")
print("  era5_mean_YYYY_MM.nc   → t2m, d2m, stl1, swvl1 at 00/06/12/18 UTC")
print("  era5_precip_YYYY_MM.nc → tp at 23:00 UTC = true daily total")

2026-05-01 16:00:19,789 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-01 16:00:19,794

2026-05-01 16:00:21,303 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-05-01 16:00:34,726 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-05-01 16:00:42,397 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


1c56a1d14bcd5b4d220b743cab77c2c5.zip:   0%|          | 0.00/88.8M [00:00<?, ?B/s]

2026-05-01 16:00:44,669 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-01 16:00:44,673

✓ Saved: era5_mean_2011_01.nc


...

2026-05-01 16:18:18,281 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-05-01 16:18:18,283

✓ Saved: era5_precip_2012_11.nc


2026-05-01 16:18:39,434 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


381625a80540795c5c440b4723b9bc3b.zip:   0%|          | 0.00/6.22M [00:00<?, ?B/s]

✓ Saved: era5_precip_2012_12.nc

✅ All PRECIP downloads done!

Summary:
  era5_mean_YYYY_MM.nc   → t2m, d2m, stl1, swvl1 at 00/06/12/18 UTC
  era5_precip_YYYY_MM.nc → tp at 23:00 UTC = true daily total


In [ ]:
import cdsapi
import os

c = cdsapi.Client()

folder_path = '/content/drive/MyDrive/project_kar'

all_24_hours = [f'{h:02d}:00' for h in range(24)]  # ['00:00', '01:00', ..., '23:00']


for year in range(2006, 2021):
    for month in range(1, 13):

        output_file = f'{folder_path}/era5_mean_{year}_{month:02d}.nc'

        if os.path.exists(output_file):
            print(f"Mean {year}-{month:02d} already exists, skipping...")
            continue

        print(f"Downloading mean variables — {year}-{month:02d}...")

        c.retrieve(
            'reanalysis-era5-land',
            {
                'variable': [
                    '2m_temperature',
                    '2m_dewpoint_temperature',
                    'soil_temperature_level_1',
                    'volumetric_soil_water_layer_1',
                ],
                'year':  [str(year)],
                'month': [f'{month:02d}'],
                'day':   [f'{d:02d}' for d in range(1, 32)],
                'time':  all_24_hours,  # all 24 hours
                'area':  [50, -125, 24, -67],
                'format': 'netcdf',
            },
            output_file
        )
        print(f"✓ Saved: era5_mean_{year}_{month:02d}.nc")

print("\n✅ All MEAN downloads done!\n")



for year in range(2006, 2021):
    for month in range(1, 13):

        output_file = f'{folder_path}/era5_precip_{year}_{month:02d}.nc'

        if os.path.exists(output_file):
            print(f"Precip {year}-{month:02d} already exists, skipping...")
            continue

        print(f"Downloading precipitation — {year}-{month:02d}...")

        c.retrieve(
            'reanalysis-era5-land',
            {
                'variable': ['total_precipitation'],
                'year':  [str(year)],
                'month': [f'{month:02d}'],
                'day':   [f'{d:02d}' for d in range(1, 32)],
                'time':  all_24_hours,
                'area':  [50, -125, 24, -67],
                'format': 'netcdf',
            },
            output_file
        )
        print(f"✓ Saved: era5_precip_{year}_{month:02d}.nc")

print("\n✅ All PRECIP downloads done!")
print("\nSummary:")
print("  era5_mean_YYYY_MM.nc   → t2m, d2m, stl1, swvl1 at all 24 hours (average daily)")
print("  era5_precip_YYYY_MM.nc → tp at all 24 hours (sum for daily total)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = '/content/drive/MyDrive/project_kar'
print("Drive mounted!")

In [ ]:
!pip install netcdf4 pykrige --quiet

import numpy as np
import pandas as pd
import scipy.stats as stats
import xarray as xr
import pickle
import os
import glob
print("Libraries loaded!")

In [ ]:
station_data = pd.read_parquet(f'{folder_path}/station_data.parquet')
print(f"✓ station_data: {station_data.shape}")
print(station_data.head())

In [ ]:
grid_lons   = np.load(f'{folder_path}/grid_lons.npy')
grid_lats   = np.load(f'{folder_path}/grid_lats.npy')
grid_lon_2d = np.load(f'{folder_path}/grid_lon_2d.npy')
grid_lat_2d = np.load(f'{folder_path}/grid_lat_2d.npy')

print(f"✓ Grid: {grid_lon_2d.shape}")
print(f"  Lons: {grid_lons[0]} to {grid_lons[-1]}")
print(f"  Lats: {grid_lats[0]} to {grid_lats[-1]}")

In [ ]:
# Load IDW
with open(f'{folder_path}/results_idw.pkl', 'rb') as f:
    results_idw = pickle.load(f)

# Load all Kriging models
with open(f'{folder_path}/results_kriging_all.pkl', 'rb') as f:
    results_kriging_all = pickle.load(f)

print(f"✓ IDW variables: {list(results_idw.keys())}")
print(f"✓ Kriging models: {list(results_kriging_all.keys())}")

In [ ]:
!pip install netcdf4 pykrige --quiet

import numpy as np
import pandas as pd
import scipy.stats as stats
import xarray as xr
import pickle
import os
import glob
print("Libraries loaded!")
import zipfile
import os
import glob
import shutil

folder_path = '/content/drive/MyDrive/project_kar'


mean_zips = sorted(glob.glob(f'{folder_path}/era5_mean_*.nc'))
print(f"\nExtracting {len(mean_zips)} mean zip files...")

for zip_path in mean_zips:

    basename = os.path.basename(zip_path)
    year_month = basename.replace('era5_mean_', '').replace('.nc', '')  # 2006_01

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            for name in z.namelist():
                # Extract to temp location
                z.extract(name, f'{folder_path}/era5_mean_extracted/')
                # Rename to include year_month
                old_path = f'{folder_path}/era5_mean_extracted/{name}'
                new_path = f'{folder_path}/era5_mean_extracted/era5_mean_{year_month}.nc'
                os.rename(old_path, new_path)
        print(f"✓ {basename} → era5_mean_{year_month}.nc")
    except Exception as e:
        print(f"✗ Failed: {basename} — {e}")

print("\n✅ All mean files extracted!\n")


precip_zips = sorted(glob.glob(f'{folder_path}/era5_precip_*.nc'))
print(f"Extracting {len(precip_zips)} precip zip files...")

for zip_path in precip_zips:
    basename  = os.path.basename(zip_path)
    year_month = basename.replace('era5_precip_', '').replace('.nc', '')

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            for name in z.namelist():
                z.extract(name, f'{folder_path}/era5_precip_extracted/')
                old_path = f'{folder_path}/era5_precip_extracted/{name}'
                new_path = f'{folder_path}/era5_precip_extracted/era5_precip_{year_month}.nc'
                os.rename(old_path, new_path)
        print(f"✓ {basename} → era5_precip_{year_month}.nc")
    except Exception as e:
        print(f"✗ Failed: {basename} — {e}")

print("\n✅ All precip files extracted!")

# ── Verify ─────────────────────────────────────────────────────────────────────
mean_nc   = sorted(glob.glob(f'{folder_path}/era5_mean_extracted/*.nc'))
precip_nc = sorted(glob.glob(f'{folder_path}/era5_precip_extracted/*.nc'))
print(f"\nExtracted mean files  : {len(mean_nc)}")
print(f"Extracted precip files: {len(precip_nc)}")
print(f"\nFirst mean file : {os.path.basename(mean_nc[0])}")
print(f"Last mean file  : {os.path.basename(mean_nc[-1])}")
print(f"First precip file: {os.path.basename(precip_nc[0])}")
print(f"Last precip file : {os.path.basename(precip_nc[-1])}")

In [ ]:
import xarray as xr
import glob


mean_files = sorted(glob.glob(f'{folder_path}/era5_mean_extracted/*.nc'))
print(f"Found {len(mean_files)} mean files")

era5_mean = xr.open_mfdataset(
    mean_files,
    combine='by_coords',
    engine='netcdf4',
    parallel=False
)

if 'valid_time' in era5_mean.dims:
    era5_mean = era5_mean.rename({'valid_time': 'time'})

print("✓ ERA5 mean loaded!")
print(f"  Variables : {list(era5_mean.data_vars)}")
print(f"  Time range: {str(era5_mean.time.values[0])[:10]} to "
      f"{str(era5_mean.time.values[-1])[:10]}")
print(f"  Shape     : {dict(era5_mean.sizes)}")


precip_files = sorted(glob.glob(f'{folder_path}/era5_precip_extracted/*.nc'))
print(f"\nFound {len(precip_files)} precip files")

era5_precip = xr.open_mfdataset(
    precip_files,
    combine='by_coords',
    engine='netcdf4',
    parallel=False
)


if 'valid_time' in era5_precip.dims:
    era5_precip = era5_precip.rename({'valid_time': 'time'})

print("✓ ERA5 precip loaded!")
print(f"  Variables : {list(era5_precip.data_vars)}")
print(f"  Time range: {str(era5_precip.time.values[0])[:10]} to "
      f"{str(era5_precip.time.values[-1])[:10]}")
print(f"  Shape     : {dict(era5_precip.sizes)}")

print("\n✅ All ERA5 files loaded successfully!")

In [ ]:
import numpy as np
import xarray as xr

print("Processing ERA5 to daily values...")

era5_daily_mean = era5_mean.resample(time='1D').mean()
print("✓ Mean variables aggregated to daily mean")
print(f"  Shape: {dict(era5_daily_mean.sizes)}")

era5_daily_mean['t2m_degC']    = era5_daily_mean['t2m'] - 273.15
era5_daily_mean['soil_temp_C'] = era5_daily_mean['stl1'] - 273.15

T_C  = era5_daily_mean['t2m'] - 273.15
Td_C = era5_daily_mean['d2m'] - 273.15
era5_daily_mean['relative_humidity'] = (
    100 * np.exp((17.625 * Td_C) / (243.04 + Td_C))
        / np.exp((17.625 * T_C)  / (243.04 + T_C))
).clip(0, 100)

print("✓ Unit conversions done!")
print("  t2m → t2m_degC (°C)")
print("  stl1 → soil_temp_C (°C)")
print("  t2m + d2m → relative_humidity (%)")

era5_daily_precip = era5_precip.copy()
era5_daily_precip['precip_mm'] = era5_daily_precip['tp'] * 1000
print("✓ Precipitation converted m → mm")

era5_mean_usa = era5_daily_mean.sel(
    latitude=slice(50, 24),
    longitude=slice(-125, -67)
)
era5_precip_usa = era5_daily_precip.sel(
    latitude=slice(50, 24),
    longitude=slice(-125, -67)
)

era5_mean_coarse = era5_mean_usa.coarsen(
    latitude=20, longitude=20, boundary='trim'
).mean()

era5_precip_coarse = era5_precip_usa.coarsen(
    latitude=20, longitude=20, boundary='trim'
).mean()

print("✓ Coarsened to 2° resolution")
print(f"  Mean shape  : {dict(era5_mean_coarse.sizes)}")
print(f"  Precip shape: {dict(era5_precip_coarse.sizes)}")

era5_mean_coarse   = era5_mean_coarse.rename({'latitude': 'lat', 'longitude': 'lon'})
era5_precip_coarse = era5_precip_coarse.rename({'latitude': 'lat', 'longitude': 'lon'})

era5_mean_2deg = era5_mean_coarse.interp(
    lat=xr.DataArray(grid_lats, dims='lat'),
    lon=xr.DataArray(grid_lons, dims='lon'),
    method='linear'
)

era5_precip_2deg = era5_precip_coarse.interp(
    lat=xr.DataArray(grid_lats, dims='lat'),
    lon=xr.DataArray(grid_lons, dims='lon'),
    method='linear'
)

print("✓ Aligned to interpolation grid")
print(f"  ERA5 lats : {era5_mean_2deg.lat.values}")
print(f"  Your lats : {grid_lats}")
print(f"  Match     : {np.allclose(era5_mean_2deg.lat.values, grid_lats)}")

print("\n✅ ERA5 processing complete!")
print(f"  Mean variables : {list(era5_mean_2deg.data_vars)}")
print(f"  Precip variable: {list(era5_precip_2deg.data_vars)}")

In [ ]:
import xarray as xr
import numpy as np
import os
import glob

print("Processing and saving ERA5 year by year...\n")

folder_path = '/content/drive/MyDrive/project_kar'

# ── Create temp folder for yearly files ───────────────────────────────────────
os.makedirs(f'{folder_path}/era5_yearly_processed', exist_ok=True)

for year in range(2006, 2022):

    # Skip if already processed
    mean_done   = os.path.exists(f'{folder_path}/era5_yearly_processed/mean_{year}.nc')
    precip_done = os.path.exists(f'{folder_path}/era5_yearly_processed/precip_{year}.nc')

    if mean_done and precip_done:
        print(f"✓ {year} already processed, skipping...")
        continue

    print(f"Processing {year}...")

    # ── Load just this year ────────────────────────────────────────────────────
    mean_files = sorted(glob.glob(
        f'{folder_path}/era5_mean_extracted/era5_mean_{year}_*.nc'
    ))
    precip_files = sorted(glob.glob(
        f'{folder_path}/era5_precip_extracted/era5_precip_{year}_*.nc'
    ))

    ds_mean = xr.open_mfdataset(
        mean_files, combine='by_coords',
        engine='netcdf4', parallel=False
    )
    if 'valid_time' in ds_mean.dims:
        ds_mean = ds_mean.rename({'valid_time': 'time'})

    ds_precip = xr.open_mfdataset(
        precip_files, combine='by_coords',
        engine='netcdf4', parallel=False
    )
    if 'valid_time' in ds_precip.dims:
        ds_precip = ds_precip.rename({'valid_time': 'time'})

    # ── Process mean ──────────────────────────────────────────────────────────
    ds_mean_daily = ds_mean.resample(time='1D').mean().compute()
    ds_mean.close()   # free memory immediately

    ds_mean_daily['t2m_degC']    = ds_mean_daily['t2m'] - 273.15
    ds_mean_daily['soil_temp_C'] = ds_mean_daily['stl1'] - 273.15

    T_C  = ds_mean_daily['t2m'] - 273.15
    Td_C = ds_mean_daily['d2m'] - 273.15
    ds_mean_daily['relative_humidity'] = (
        100 * np.exp((17.625 * Td_C) / (243.04 + Td_C))
            / np.exp((17.625 * T_C)  / (243.04 + T_C))
    ).clip(0, 100)

    ds_mean_usa = ds_mean_daily.sel(
        latitude=slice(50, 24), longitude=slice(-125, -67)
    )
    ds_mean_coarse = ds_mean_usa.coarsen(
        latitude=20, longitude=20, boundary='trim'
    ).mean()
    ds_mean_coarse = ds_mean_coarse.rename({'latitude': 'lat', 'longitude': 'lon'})
    ds_mean_2deg = ds_mean_coarse.interp(
        lat=xr.DataArray(grid_lats, dims='lat'),
        lon=xr.DataArray(grid_lons, dims='lon'),
        method='linear'
    )

    # ── Save mean immediately — do NOT keep in RAM ────────────────────────────
    ds_mean_2deg.to_netcdf(
        f'{folder_path}/era5_yearly_processed/mean_{year}.nc',
        encoding={var: {'zlib': True, 'complevel': 1}
                  for var in ds_mean_2deg.data_vars}
    )
    del ds_mean_daily, ds_mean_usa, ds_mean_coarse, ds_mean_2deg
    print(f"  ✓ Mean {year} saved")

    # ── Process precip ────────────────────────────────────────────────────────
    ds_precip_loaded = ds_precip.compute()
    ds_precip.close()

    ds_precip_loaded['precip_mm'] = ds_precip_loaded['tp'] * 1000
    ds_precip_usa = ds_precip_loaded.sel(
        latitude=slice(50, 24), longitude=slice(-125, -67)
    )
    ds_precip_coarse = ds_precip_usa.coarsen(
        latitude=20, longitude=20, boundary='trim'
    ).mean()
    ds_precip_coarse = ds_precip_coarse.rename({'latitude': 'lat', 'longitude': 'lon'})
    ds_precip_2deg = ds_precip_coarse.interp(
        lat=xr.DataArray(grid_lats, dims='lat'),
        lon=xr.DataArray(grid_lons, dims='lon'),
        method='linear'
    )

    # ── Save precip immediately ────────────────────────────────────────────────
    ds_precip_2deg.to_netcdf(
        f'{folder_path}/era5_yearly_processed/precip_{year}.nc',
        encoding={var: {'zlib': True, 'complevel': 1}
                  for var in ds_precip_2deg.data_vars}
    )
    del ds_precip_loaded, ds_precip_usa, ds_precip_coarse, ds_precip_2deg
    print(f"  ✓ Precip {year} saved")

print("\n✅ All years processed and saved!")

# ── Combine all yearly files into final files ─────────────────────────────────
print("\nCombining all years into final files...")

mean_yearly   = sorted(glob.glob(f'{folder_path}/era5_yearly_processed/mean_*.nc'))
precip_yearly = sorted(glob.glob(f'{folder_path}/era5_yearly_processed/precip_*.nc'))

era5_mean_final = xr.open_mfdataset(
    mean_yearly, combine='by_coords', engine='netcdf4'
)
era5_mean_final.to_netcdf(
    f'{folder_path}/era5_mean_2deg_daily.nc',
    encoding={var: {'zlib': True, 'complevel': 1}
              for var in era5_mean_final.data_vars}
)
print("✓ Saved: era5_mean_2deg_daily.nc")

era5_precip_final = xr.open_mfdataset(
    precip_yearly, combine='by_coords', engine='netcdf4'
)
era5_precip_final.to_netcdf(
    f'{folder_path}/era5_precip_2deg_daily.nc',
    encoding={var: {'zlib': True, 'complevel': 1}
              for var in era5_precip_final.data_vars}
)
print("✓ Saved: era5_precip_2deg_daily.nc")

print("\n✅ Final files ready!")

In [ ]:
import numpy as np
import scipy.stats as stats

var_mapping = {
    'air_temperature':   ('mean',   't2m_degC'),
    'precipitation':     ('precip', 'precip_mm'),
    'relative_humidity': ('mean',   'relative_humidity'),
    'soil_temperature':  ('mean',   'soil_temp_C'),
    'soil_moisture':     ('mean',   'swvl1'),
}

season_order = ['DJF (Winter)', 'MAM (Spring)', 'JJA (Summer)', 'SON (Autumn)']
seasons = [get_season(d.month) for d in date_range]

methods = {
    'IDW':                 results_idw,
    'Kriging_Spherical':   results_kriging_all['spherical'],
    'Kriging_Exponential': results_kriging_all['exponential'],
    'Kriging_Gaussian':    results_kriging_all['gaussian'],
}

for var, (source, era5_var) in var_mapping.items():

    if source == 'mean':
        era5_vals_full = era5_mean_2deg[era5_var].values
    else:
        era5_vals_full = era5_precip_2deg[era5_var].values

    print(f"\n{'='*70}")
    print(f"VARIABLE: {var}")
    print(f"{'='*70}")

    # ── Overall statistics (full period) ──────────────────────────────────────
    print(f"\n--- OVERALL (Full Period 2006-2021) ---")
    print(f"{'Method':22s} | {'Mean':>8} | {'Median':>8} | {'Std':>8} | {'Variance':>10} | {'Min':>8} | {'Max':>8}")
    print("-" * 85)

    for method_name, results in methods.items():
        arr = results[var].ravel()
        arr = arr[~np.isnan(arr)]
        print(f"{method_name:22s} | {np.mean(arr):>8.4f} | {np.median(arr):>8.4f} | "
              f"{np.std(arr):>8.4f} | {np.var(arr):>10.4f} | "
              f"{np.min(arr):>8.4f} | {np.max(arr):>8.4f}")

    era5_arr = era5_vals_full.ravel()
    era5_arr = era5_arr[~np.isnan(era5_arr)]
    print(f"{'ERA5':22s} | {np.mean(era5_arr):>8.4f} | {np.median(era5_arr):>8.4f} | "
          f"{np.std(era5_arr):>8.4f} | {np.var(era5_arr):>10.4f} | "
          f"{np.min(era5_arr):>8.4f} | {np.max(era5_arr):>8.4f}")

    # ── Performance metrics (full period) ─────────────────────────────────────
    print(f"\n--- PERFORMANCE vs ERA5 (Full Period) ---")
    print(f"{'Method':22s} | {'RMSE':>8} | {'r':>8} | {'KS D':>8} | {'Similarity':>20}")
    print("-" * 75)

    for method_name, results in methods.items():
        pred = results[var].ravel()
        ref  = era5_vals_full.ravel()
        mask = ~(np.isnan(pred) | np.isnan(ref))
        pred_c = pred[mask]; ref_c = ref[mask]
        rmse = np.sqrt(np.mean((pred_c - ref_c)**2))
        r, _  = stats.pearsonr(pred_c, ref_c)
        d, _  = stats.ks_2samp(pred_c, ref_c)
        sim = 'Very similar' if d<0.05 else 'Slight diff' if d<0.15 else 'Moderate diff' if d<0.30 else 'Large diff'
        print(f"{method_name:22s} | {rmse:>8.4f} | {r:>8.4f} | {d:>8.4f} | {sim:>20}")

    # ── Seasonal breakdown ────────────────────────────────────────────────────
    for season in season_order:
        s_idx = [i for i, s in enumerate(seasons) if s == season]

        print(f"\n--- {season} ---")
        print(f"{'Method':22s} | {'Mean':>8} | {'Median':>8} | {'Std':>8} | {'Variance':>10} | {'Min':>8} | {'Max':>8}")
        print("-" * 85)

        for method_name, results in methods.items():
            arr = results[var][s_idx].ravel()
            arr = arr[~np.isnan(arr)]
            print(f"{method_name:22s} | {np.mean(arr):>8.4f} | {np.median(arr):>8.4f} | "
                  f"{np.std(arr):>8.4f} | {np.var(arr):>10.4f} | "
                  f"{np.min(arr):>8.4f} | {np.max(arr):>8.4f}")

        era5_s = era5_vals_full[s_idx].ravel()
        era5_s = era5_s[~np.isnan(era5_s)]
        print(f"{'ERA5':22s} | {np.mean(era5_s):>8.4f} | {np.median(era5_s):>8.4f} | "
              f"{np.std(era5_s):>8.4f} | {np.var(era5_s):>10.4f} | "
              f"{np.min(era5_s):>8.4f} | {np.max(era5_s):>8.4f}")

        print(f"\n{'Method':22s} | {'RMSE':>8} | {'r':>8} | {'KS D':>8} | {'Similarity':>20}")
        print("-" * 75)

        for method_name, results in methods.items():
            pred = results[var][s_idx].ravel()
            ref  = era5_vals_full[s_idx].ravel()
            mask = ~(np.isnan(pred) | np.isnan(ref))
            pred_c = pred[mask]; ref_c = ref[mask]
            rmse = np.sqrt(np.mean((pred_c - ref_c)**2))
            r, _  = stats.pearsonr(pred_c, ref_c)
            d, _  = stats.ks_2samp(pred_c, ref_c)
            sim = 'Very similar' if d<0.05 else 'Slight diff' if d<0.15 else 'Moderate diff' if d<0.30 else 'Large diff'
            print(f"{method_name:22s} | {rmse:>8.4f} | {r:>8.4f} | {d:>8.4f} | {sim:>20}")

In [ ]:
"""
GNR 640 — Complete Figure Generation Code
Run this in your Colab notebook after loading all variables.
All figures saved at 360 DPI in organized subfolders.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import pearsonr
import os
import warnings
warnings.filterwarnings('ignore')

# ── Folder structure ───────────────────────────────────────────────────────────
BASE = f'{folder_path}/figures'
FOLDERS = {
    'data':        f'{BASE}/01_data_sources',
    'method':      f'{BASE}/02_methodology',
    'performance': f'{BASE}/03_performance',
    'stats':       f'{BASE}/04_statistical_analysis',
    'seasonal':    f'{BASE}/05_seasonality',
}
for f in FOLDERS.values():
    os.makedirs(f, exist_ok=True)

DPI    = 360
FSIZE  = (12, 7)

# Consistent color palette
METHOD_COLORS = {
    'IDW':                '#2166AC',
    'Kriging Spherical':   '#4DAC26',
    'Kriging Exponential': '#D7191C',
    'Kriging Gaussian':    '#F4A582',
    'ERA5':                '#636363',
}
SEASON_COLORS  = ['#2C7BB6', '#5AAE61', '#D73027', '#F4A742']
SEASON_LABELS  = ['DJF (Winter)', 'MAM (Spring)', 'JJA (Summer)', 'SON (Autumn)']
VARIABLES      = ['air_temperature', 'precipitation', 'relative_humidity',
                  'soil_temperature', 'soil_moisture']
VAR_LABELS     = ['Air Temperature', 'Precipitation',
                  'Relative Humidity', 'Soil Temperature', 'Soil Moisture']
VAR_UNITS      = ['°C', 'mm/day', '%', '°C', 'm³/m³']
VAR_CMAPS      = ['RdYlBu_r', 'YlGnBu', 'BuGn', 'RdYlBu_r', 'YlOrBr_r']

plt.rcParams.update({
    'font.family':     'DejaVu Sans',
    'font.size':       11,
    'axes.titlesize':  13,
    'axes.labelsize':  11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi':      100,
})

def save(fig, folder, filename):
    path = os.path.join(FOLDERS[folder], filename)
    fig.savefig(path, dpi=DPI, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig)
    print(f'  ✓ Saved: {filename}')

def get_season(month):
    if month in [12, 1, 2]:  return 'DJF (Winter)'
    elif month in [3, 4, 5]: return 'MAM (Spring)'
    elif month in [6, 7, 8]: return 'JJA (Summer)'
    else:                    return 'SON (Autumn)'

import pandas as pd
date_range_local = pd.date_range('2006-01-01', '2021-12-31', freq='D')
seasons_list = [get_season(d.month) for d in date_range_local]

ERA5_VARS = {
    'air_temperature':   ('mean',   't2m_degC'),
    'precipitation':     ('precip', 'precip_mm'),
    'relative_humidity': ('mean',   'relative_humidity'),
    'soil_temperature':  ('mean',   'soil_temp_C'),
    'soil_moisture':     ('mean',   'swvl1'),
}

def get_era5(var):
    source, key = ERA5_VARS[var]
    if source == 'mean':
        return era5_mean_2deg[key].values
    return era5_precip_2deg[key].values


print('=' * 60)
print('GNR 640 — Generating all figures at 360 DPI')
print('=' * 60)


# ═══════════════════════════════════════════════════════════
# SECTION 1.2 — DATA SOURCES
# ═══════════════════════════════════════════════════════════
print('\n[1/5] Section 1.2 — Data Sources')

# Fig 1.2.1 — USCRN Station Location Map
# Fig 1.2.1 — USCRN Station Location Map
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_facecolor('#D6EAF8')
ax.set_xlim(-130, -64); ax.set_ylim(22, 52)
ax.set_xlabel('Longitude (°)', fontsize=11)
ax.set_ylabel('Latitude (°)',  fontsize=11)

# FIX: use paired lon/lat per station, not separate unique values
station_coords = station_data[['lon', 'lat']].drop_duplicates()
lons_s = station_coords['lon'].values
lats_s = station_coords['lat'].values

ax.scatter(lons_s, lats_s,
           c='#D73027', s=35, marker='^', zorder=5,
           edgecolors='#7B0000', linewidths=0.4,
           label=f'USCRN Stations (n={len(lons_s)})')

ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_title('Fig. 1.2.1  USCRN Station Network — Contiguous United States\n'
             'Station locations used as input for spatial interpolation (2006–2021)',
             pad=12, fontsize=12)
save(fig, 'data', 'Fig1_2_1_USCRN_Station_Locations.png')


# ═══════════════════════════════════════════════════════════
# SECTION 1.3 — METHODOLOGY
# ═══════════════════════════════════════════════════════════
print('\n[2/5] Section 1.3 — Methodology')

# Fig 1.3.1 — 2-degree interpolation grid
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_facecolor('#D6EAF8')
ax.set_xlim(-128, -64); ax.set_ylim(22, 52)
ax.set_xlabel('Longitude (°)', fontsize=11)
ax.set_ylabel('Latitude (°)',  fontsize=11)
for lon in grid_lons:
    ax.axvline(lon, color='#2166AC', lw=0.6, alpha=0.5)
for lat in grid_lats:
    ax.axhline(lat, color='#2166AC', lw=0.6, alpha=0.5)
ax.scatter([l for l in grid_lons for _ in grid_lats],
           [la for _ in grid_lons for la in grid_lats],
           c='#2166AC', s=40, zorder=4, label='Grid node (2° × 2°)')
# FIX: use paired coords here too
station_coords = station_data[['lon', 'lat']].drop_duplicates()
lons_s = station_coords['lon'].values
lats_s = station_coords['lat'].values
ax.scatter(lons_s, lats_s, c='#D73027', s=20, marker='^', zorder=5,
           edgecolors='#7B0000', linewidths=0.3, label='USCRN station')
ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.grid(False)
ax.set_title('Fig. 1.3.1  2-Degree Interpolation Grid over Contiguous USA\n'
             f'Grid: {len(grid_lats)} lat × {len(grid_lons)} lon = '
             f'{len(grid_lats)*len(grid_lons)} grid cells',
             pad=12, fontsize=12)
save(fig, 'method', 'Fig1_3_1_Interpolation_Grid.png')

# Fig 1.3.2 — Variogram model comparison curves
fig, ax = plt.subplots(figsize=(10, 6))
h = np.linspace(0, 1, 500)
nugget, sill, a = 0.05, 1.0, 0.6
gamma_sph = np.where(h <= a,
    nugget + (sill-nugget)*(1.5*(h/a) - 0.5*(h/a)**3),
    sill)
gamma_exp = nugget + (sill-nugget)*(1 - np.exp(-3*h/a))
gamma_gau = nugget + (sill-nugget)*(1 - np.exp(-3*(h/a)**2))
ax.plot(h, gamma_sph, color='#4DAC26', lw=2.5, label='Spherical')
ax.plot(h, gamma_exp, color='#D7191C', lw=2.5, linestyle='--', label='Exponential')
ax.plot(h, gamma_gau, color='#F4A582', lw=2.5, linestyle=':', label='Gaussian')
ax.axhline(sill,   color='gray', lw=1, ls='--', alpha=0.6)
ax.axhline(nugget, color='gray', lw=1, ls='--', alpha=0.6)
ax.axvline(a,      color='gray', lw=1, ls='--', alpha=0.6)
ax.text(1.02, sill,   'Sill',   va='center', fontsize=10, color='gray')
ax.text(1.02, nugget, 'Nugget', va='center', fontsize=10, color='gray')
ax.text(a, -0.08, 'Range (a)', ha='center', fontsize=10, color='gray')
ax.set_xlabel('Lag distance (h)', fontsize=11)
ax.set_ylabel('Semivariance γ(h)', fontsize=11)
ax.legend(fontsize=11, framealpha=0.9)
ax.set_ylim(-0.05, 1.15)
ax.set_title('Fig. 1.3.2  Theoretical Variogram Models Used in Ordinary Kriging\n'
             'Spherical, Exponential, and Gaussian models — normalised sill = 1.0',
             pad=12, fontsize=12)
save(fig, 'method', 'Fig1_3_2_Variogram_Models.png')


# ═══════════════════════════════════════════════════════════
# SECTION 1.4 — PERFORMANCE EVALUATION
# ═══════════════════════════════════════════════════════════
print('\n[3/5] Section 1.4 — Performance Evaluation')

# Fig 1.4.1 — RMSE grouped bar chart
rmse_vals = {
    'IDW':                [3.2023, 4.2132, 11.167, 4.4810, 0.0957],
    'Kriging Spherical':   [2.8568, 4.2347, 10.939, 4.4113, 0.0939],
    'Kriging Exponential': [3.2737, 4.5166, 11.017, 4.5096, 0.0967],
    'Kriging Gaussian':    [4.6263, 3.7159, 12.880, 5.0081, 0.0978],
}
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(5); w = 0.2
for i, (method, vals) in enumerate(rmse_vals.items()):
    bars = ax.bar(x + i*w - 0.3, vals, w, label=method,
                  color=list(METHOD_COLORS.values())[i],
                  edgecolor='white', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n({u})' for l, u in zip(VAR_LABELS, VAR_UNITS)],
                   fontsize=10)
ax.set_ylabel('RMSE', fontsize=11)
ax.legend(fontsize=10, framealpha=0.9, ncol=2)
ax.set_title('Fig. 1.4.1  RMSE Comparison — All Interpolation Methods and Variables\n'
             'Lower RMSE indicates better agreement with ERA5-Land reference data',
             pad=12, fontsize=12)
save(fig, 'performance', 'Fig1_4_1_RMSE_Comparison.png')

# Fig 1.4.2 — Correlation grouped bar chart
corr_vals = {
    'IDW':                [0.9624, 0.6478, 0.8318, 0.9296, 0.6607],
    'Kriging Spherical':   [0.9699, 0.6293, 0.8405, 0.9379, 0.6835],
    'Kriging Exponential': [0.9602, 0.4417, 0.8391, 0.9319, 0.6502],
    'Kriging Gaussian':    [0.9248, 0.6501, 0.7912, 0.8964, 0.6323],
}
fig, ax = plt.subplots(figsize=(14, 6))
for i, (method, vals) in enumerate(corr_vals.items()):
    ax.bar(x + i*w - 0.3, vals, w, label=method,
           color=list(METHOD_COLORS.values())[i],
           edgecolor='white', linewidth=0.5)
ax.axhline(1.0, color='gray', lw=0.8, ls='--', alpha=0.5)
ax.set_ylim(0, 1.08)
ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n({u})' for l, u in zip(VAR_LABELS, VAR_UNITS)],
                   fontsize=10)
ax.set_ylabel('Pearson Correlation (r)', fontsize=11)
ax.legend(fontsize=10, framealpha=0.9, ncol=2)
ax.set_title('Fig. 1.4.2  Pearson Correlation Comparison — All Interpolation Methods\n'
             'Higher r indicates stronger linear agreement with ERA5-Land reference',
             pad=12, fontsize=12)
save(fig, 'performance', 'Fig1_4_2_Correlation_Comparison.png')

# Fig 1.4.3-1.4.7 — Scatter plots (IDW + Kriging Spherical vs ERA5)
for vi, var in enumerate(VARIABLES):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    era5 = get_era5(var).ravel()
    for ai, (method_name, results) in enumerate([
        ('IDW',               results_idw),
        ('Kriging Spherical', results_kriging_all['spherical']),
    ]):
        ax   = axes[ai]
        pred = results[var].ravel()
        mask = ~(np.isnan(pred) | np.isnan(era5))
        p_c, e_c = pred[mask], era5[mask]
        # Sample 50k for plotting speed
        if len(p_c) > 50000:
            idx = np.random.choice(len(p_c), 50000, replace=False)
            p_c, e_c = p_c[idx], e_c[idx]
        r, _ = pearsonr(p_c, e_c)
        rmse = np.sqrt(np.mean((p_c - e_c)**2))
        ax.scatter(e_c, p_c, s=1, alpha=0.15, color=list(METHOD_COLORS.values())[ai],
                   rasterized=True)
        lims = [min(e_c.min(), p_c.min()), max(e_c.max(), p_c.max())]
        ax.plot(lims, lims, 'k--', lw=1, label='1:1 line')
        ax.set_xlabel(f'ERA5 ({VAR_UNITS[vi]})', fontsize=11)
        ax.set_ylabel(f'{method_name} ({VAR_UNITS[vi]})', fontsize=11)
        ax.set_title(f'{method_name}', fontsize=12, fontweight='bold')
        ax.text(0.05, 0.92, f'r = {r:.4f}\nRMSE = {rmse:.4f} {VAR_UNITS[vi]}',
                transform=ax.transAxes, fontsize=10,
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))
        ax.legend(fontsize=9)
    fig.suptitle(f'Fig. 1.4.{vi+3}  Scatter Plot — {VAR_LABELS[vi]} '
                 f'vs ERA5-Land Reference\nIDW (left) and Kriging Spherical (right) '
                 f'— full 2006–2021 period (n ≈ 2.2M grid-day values)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    save(fig, 'performance',
         f'Fig1_4_{vi+3}_Scatter_{var.title().replace("_","")}.png')


# ═══════════════════════════════════════════════════════════
# SECTION 1.5 — STATISTICAL ANALYSIS
# ═══════════════════════════════════════════════════════════
print('\n[4/5] Section 1.5 — Statistical Analysis')

# Fig 1.5.1 — Box plots (IDW vs ERA5) for all variables
fig, axes = plt.subplots(1, 5, figsize=(20, 7))
for vi, var in enumerate(VARIABLES):
    ax   = axes[vi]
    era5 = get_era5(var).ravel()
    era5 = era5[~np.isnan(era5)]
    idw  = results_idw[var].ravel()
    idw  = idw[~np.isnan(idw)]
    if len(era5) > 100000:
        era5 = np.random.choice(era5, 100000, replace=False)
        idw  = np.random.choice(idw,  100000, replace=False)
    bp = ax.boxplot([idw, era5],
                    labels=['IDW', 'ERA5'],
                    patch_artist=True,
                    medianprops=dict(color='black', lw=2),
                    flierprops=dict(marker='.', ms=1, alpha=0.2),
                    whiskerprops=dict(lw=1.2),
                    capprops=dict(lw=1.2))
    bp['boxes'][0].set_facecolor(METHOD_COLORS['IDW'] + '88')
    bp['boxes'][1].set_facecolor(METHOD_COLORS['ERA5'] + '88')
    ax.set_title(f'{VAR_LABELS[vi]}\n({VAR_UNITS[vi]})', fontsize=10, fontweight='bold')
    ax.set_xlabel('')
fig.suptitle('Fig. 1.5.1  Box Plot Comparison — IDW Interpolated vs ERA5-Land\n'
             'Showing median, interquartile range, and outliers (full 2006–2021 dataset)',
             fontsize=12, y=1.01)
plt.tight_layout()
save(fig, 'stats', 'Fig1_5_1_Boxplots_IDW_vs_ERA5.png')

# Fig 1.5.2 — CDF comparison (KS test visualisation) for all variables
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()
for vi, var in enumerate(VARIABLES):
    ax   = axes[vi]
    era5 = get_era5(var).ravel(); era5 = era5[~np.isnan(era5)]
    idw  = results_idw[var].ravel(); idw  = idw[~np.isnan(idw)]
    if len(era5) > 50000:
        era5 = np.random.choice(era5, 50000, replace=False)
        idw  = np.random.choice(idw,  50000, replace=False)
    era5_s = np.sort(era5); idw_s = np.sort(idw)
    ax.plot(era5_s, np.linspace(0, 1, len(era5_s)),
            color=METHOD_COLORS['ERA5'], lw=2, label='ERA5')
    ax.plot(idw_s,  np.linspace(0, 1, len(idw_s)),
            color=METHOD_COLORS['IDW'],  lw=2, ls='--', label='IDW')
    ax.set_xlabel(VAR_UNITS[vi], fontsize=10)
    ax.set_ylabel('Cumulative probability', fontsize=10)
    ax.set_title(f'{VAR_LABELS[vi]}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
axes[5].set_visible(False)
fig.suptitle('Fig. 1.5.2  Empirical CDF Comparison — IDW vs ERA5-Land\n'
             'Basis for Kolmogorov-Smirnov two-sample test; D = max vertical distance',
             fontsize=12, y=1.01)
plt.tight_layout()
save(fig, 'stats', 'Fig1_5_2_CDF_Comparison_IDW_ERA5.png')

# Fig 1.5.3 — KS D-statistic heatmap
ks_matrix = np.array([
    [0.0300, 0.2184, 0.0506, 0.1020, 0.1960],
    [0.0334, 0.2320, 0.0667, 0.1150, 0.2604],
    [0.0355, 0.3184, 0.0642, 0.1092, 0.2518],
    [0.0809, 0.4413, 0.1384, 0.0787, 0.3059],
])
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(ks_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.5)
plt.colorbar(im, ax=ax, label='KS D-statistic (0 = identical, 0.5 = large difference)')
ax.set_xticks(range(5))
ax.set_xticklabels([f'{l}\n({u})' for l, u in zip(VAR_LABELS, VAR_UNITS)], fontsize=10)
ax.set_yticks(range(4))
ax.set_yticklabels(['IDW', 'Kriging Spherical',
                    'Kriging Exponential', 'Kriging Gaussian'], fontsize=10)
for i in range(4):
    for j in range(5):
        txt  = f'{ks_matrix[i,j]:.4f}'
        col  = 'white' if ks_matrix[i,j] > 0.3 else 'black'
        ax.text(j, i, txt, ha='center', va='center', fontsize=10,
                color=col, fontweight='bold')
ax.set_title('Fig. 1.5.3  KS Test D-Statistic Heatmap — All Methods vs ERA5-Land\n'
             'Green = similar distribution, Red = large distributional difference',
             pad=12, fontsize=12)
plt.tight_layout()
save(fig, 'stats', 'Fig1_5_3_KS_Dstatistic_Heatmap.png')


# ═══════════════════════════════════════════════════════════
# SECTION 1.6 — SEASONALITY
# ═══════════════════════════════════════════════════════════
print('\n[5/5] Section 1.6 — Seasonality')

# Fig 1.6.1-1.6.5 — Seasonal mean bar charts (all methods + ERA5)
for vi, var in enumerate(VARIABLES):
    era5_arr = get_era5(var)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=False)
    for si, season in enumerate(SEASON_LABELS):
        ax    = axes[si]
        s_idx = [i for i, s in enumerate(seasons_list) if s == season]
        means = []
        labels = []
        colors = []
        for method_name, res in [
            ('IDW',                 results_idw),
            ('Kriging\nSpherical',  results_kriging_all['spherical']),
            ('Kriging\nExponent.',  results_kriging_all['exponential']),
            ('Kriging\nGaussian',   results_kriging_all['gaussian']),
            ('ERA5',                None),
        ]:
            if method_name == 'ERA5':
                arr = era5_arr[s_idx].ravel()
            else:
                arr = res[var][s_idx].ravel()
            arr = arr[~np.isnan(arr)]
            means.append(np.mean(arr))
            labels.append(method_name)
            key = method_name.replace('\n', ' ').replace('Exponent.', 'Exponential')
            colors.append(METHOD_COLORS.get(key, '#888888'))
        bars = ax.bar(range(5), means, color=colors,
                      edgecolor='white', linewidth=0.5, width=0.7)
        for bar, val in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + abs(bar.get_height())*0.01,
                    f'{val:.2f}', ha='center', va='bottom',
                    fontsize=7.5, fontweight='bold')
        ax.set_xticks(range(5))
        ax.set_xticklabels(labels, fontsize=8.5)
        ax.set_title(season, fontsize=11, fontweight='bold',
                     color=SEASON_COLORS[si])
        if si == 0:
            ax.set_ylabel(f'Mean {VAR_LABELS[vi]} ({VAR_UNITS[vi]})', fontsize=10)
    fig.suptitle(f'Fig. 1.6.{vi+1}  Seasonal Mean — {VAR_LABELS[vi]}\n'
                 f'All interpolation methods vs ERA5-Land reference — 2006–2021',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    save(fig, 'seasonal',
         f'Fig1_6_{vi+1}_SeasonalMean_{var.title().replace("_","")}.png')

# Fig 1.6.6-1.6.10 — Seasonal spatial maps (IDW, 4 seasons)
season_dates = {
    'DJF (Winter)': '2010-01-15',
    'MAM (Spring)': '2010-04-15',
    'JJA (Summer)': '2010-07-15',
    'SON (Autumn)': '2010-10-15',
}
for vi, var in enumerate(VARIABLES):
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    idw_maps  = []
    for si, (season, date_str) in enumerate(season_dates.items()):
        t_idx = list(date_range_local).index(pd.Timestamp(date_str))
        idw_maps.append(results_idw[var][t_idx])
    vmin = np.nanpercentile(np.concatenate([m.ravel() for m in idw_maps]), 2)
    vmax = np.nanpercentile(np.concatenate([m.ravel() for m in idw_maps]), 98)
    for si, (season, date_str) in enumerate(season_dates.items()):
        ax = axes[si]
        pcm = ax.pcolormesh(grid_lons, grid_lats, idw_maps[si],
                            cmap=VAR_CMAPS[vi], shading='auto',
                            vmin=vmin, vmax=vmax)
        plt.colorbar(pcm, ax=ax, label=VAR_UNITS[vi], shrink=0.85, pad=0.02)
        ax.set_title(season, fontsize=11, fontweight='bold',
                     color=SEASON_COLORS[si])
        ax.set_xlabel('Longitude', fontsize=9)
        ax.set_ylabel('Latitude',  fontsize=9)
        ax.set_xlim([-130, -60]); ax.set_ylim([22, 52])
        ax.grid(True, ls='--', alpha=0.3)
        ax.text(0.02, 0.96, date_str, transform=ax.transAxes,
                fontsize=8, va='top', style='italic')
    fig.suptitle(f'Fig. 1.6.{vi+6}  Seasonal Spatial Maps — {VAR_LABELS[vi]} (IDW)\n'
                 f'Representative days per season — shared colour scale '
                 f'({vmin:.2f}–{vmax:.2f} {VAR_UNITS[vi]})',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    save(fig, 'seasonal',
         f'Fig1_6_{vi+6}_SeasonalMaps_IDW_{var.title().replace("_","")}.png')

# Fig 1.6.11-1.6.15 — Method comparison maps (one representative summer day)
target_date = pd.Timestamp('2010-07-15')
t_idx = list(date_range_local).index(target_date)

for vi, var in enumerate(VARIABLES):
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    maps = [
        results_idw[var][t_idx],
        results_kriging_all['spherical'][var][t_idx],
        results_kriging_all['exponential'][var][t_idx],
        results_kriging_all['gaussian'][var][t_idx],
    ]
    names = ['IDW', 'Kriging Spherical', 'Kriging Exponential', 'Kriging Gaussian']
    vmin  = np.nanpercentile(np.concatenate([m.ravel() for m in maps]), 2)
    vmax  = np.nanpercentile(np.concatenate([m.ravel() for m in maps]), 98)
    for ai, (m, name) in enumerate(zip(maps, names)):
        ax  = axes[ai]
        pcm = ax.pcolormesh(grid_lons, grid_lats, m,
                            cmap=VAR_CMAPS[vi], shading='auto',
                            vmin=vmin, vmax=vmax)
        plt.colorbar(pcm, ax=ax, label=VAR_UNITS[vi], shrink=0.85, pad=0.02)
        ax.set_title(name, fontsize=10, fontweight='bold')
        ax.set_xlabel('Longitude', fontsize=9)
        ax.set_ylabel('Latitude',  fontsize=9)
        ax.set_xlim([-130, -60]); ax.set_ylim([22, 52])
        ax.grid(True, ls='--', alpha=0.3)
    fig.suptitle(f'Fig. 1.6.{vi+11}  Method Comparison Spatial Maps — '
                 f'{VAR_LABELS[vi]} ({target_date.date()})\n'
                 f'All four interpolation methods — shared colour scale '
                 f'({vmin:.2f}–{vmax:.2f} {VAR_UNITS[vi]})',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    save(fig, 'seasonal',
         f'Fig1_6_{vi+11}_MethodComparison_{var.title().replace("_","")}.png')


print('\n' + '=' * 60)
print('ALL FIGURES SAVED!')
print(f'Location: {BASE}/')
print('=' * 60)
print('\nSubfolders:')
for k, v in FOLDERS.items():
    files = os.listdir(v)
    print(f'  {v.split("/")[-1]}/  ({len(files)} figures)')